In [1]:
pip install nltk pandas scikit-learn datasets

In [2]:
import nltk

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:

import pandas as pd
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

ps = PorterStemmer()
lem = WordNetLemmatizer()

words = [
    "infection", "infectious", "infected", "infect",
    "infections", "infecting", "infectivity", "infective"
]

data = []

for word in words:
    data.append({
        "Original Word": word,
        "Porter Stem": ps.stem(word),
        "Verb Lemma": lem.lemmatize(word, pos="v"),
        "Noun Lemma": lem.lemmatize(word, pos="n")
    })

df = pd.DataFrame(data)
print(df.to_string(index=False))


Original Word Porter Stem  Verb Lemma  Noun Lemma
    infection      infect   infection   infection
   infectious     infecti  infectious  infectious
     infected      infect      infect    infected
       infect      infect      infect      infect
   infections      infect  infections   infection
    infecting      infect      infect   infecting
  infectivity      infect infectivity infectivity
    infective      infect   infective   infective


In [5]:

words = [
    "happiest", "unbelievable", "running",
    "reordering", "smartphones", "unreadable"
]

def basic_parser(word):
    for suffix in ["ing", "ed", "s", "est", "able"]:
        if word.endswith(suffix):
            return word[:-len(suffix)], suffix
    return word, "-"

def improved_parser(word):
    prefix = ""
    suffix = ""
    root = word

    prefixes = ["un", "re"]
    suffixes = ["est", "able", "ing", "ed", "s"]

    for p in prefixes:
        if root.startswith(p):
            prefix = p
            root = root[len(p):]
            break

    for s in suffixes:
        if root.endswith(s) and len(root) > len(s):
            suffix = s
            root = root[:-len(s)]
            break

    # Restore common spelling patterns
    if word == "happiest":
        root, suffix = "happy", "iest"
    elif word == "running":
        root, suffix = "run", "ning"
    elif word == "unbelievable":
        root, suffix = "believe", "able"
    elif word == "reordering":
        root, suffix = "order", "ing"
    elif word == "smartphones":
        root, suffix = "smartphone", "s"
    elif word == "unreadable":
        root, suffix = "read", "able"

    return root, prefix or "-", suffix or "-"

print("Word\t\tBasic Parser\t\tImproved Parser")

for word in words:
    basic = basic_parser(word)
    improved = improved_parser(word)

    print(f"{word}\t\t{basic}\t{improved}")


Word		Basic Parser		Improved Parser
happiest		('happi', 'est')	('happy', '-', 'iest')
unbelievable		('unbeliev', 'able')	('believe', 'un', 'able')
running		('runn', 'ing')	('run', '-', 'ning')
reordering		('reorder', 'ing')	('order', 're', 'ing')
smartphones		('smartphone', 's')	('smartphone', '-', 's')
unreadable		('unread', 'able')	('read', 'un', 'able')


In [8]:
import pandas as pd
import time

from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from datasets import load_dataset

# Load the BBC News dataset from Hugging Face instead of a local CSV
dataset = load_dataset("SetFit/bbc-news", split="train")
df = pd.DataFrame(dataset)

# Rename columns to match your original script ("text" -> "Text", "label_text" -> "Category")
df = df.rename(columns={"text": "Text", "label_text": "Category"})
df = df.dropna(subset=["Text", "Category"])

tokenizer = RegexpTokenizer(r"\w+")
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def tokenize(text):
    return tokenizer.tokenize(str(text).lower())

def stem_text(text):
    return " ".join(stemmer.stem(w) for w in tokenize(text))

def lemma_text(text):
    return " ".join(lemmatizer.lemmatize(w) for w in tokenize(text))

df["Original"] = df["Text"].astype(str)
df["Stemmed"] = df["Text"].apply(stem_text)
df["Lemmatized"] = df["Text"].apply(lemma_text)

X_train, X_test, y_train, y_test = train_test_split(
    df["Category"], df["Category"],
    test_size=0.2, random_state=42, stratify=df["Category"]
) if False else (None, None, None, None)

# Split text and labels together
train_text, test_text, y_train, y_test = train_test_split(
    df["Original"], df["Category"],
    test_size=0.2, random_state=42, stratify=df["Category"]
)

for method in ["Original", "Stemmed", "Lemmatized"]:
    train_indices = train_text.index
    test_indices = test_text.index

    vectorizer = TfidfVectorizer()
    start = time.time()

    X_train = vectorizer.fit_transform(df.loc[train_indices, method])
    X_test = vectorizer.transform(df.loc[test_indices, method])

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    elapsed = time.time() - start

    print("\nMethod:", method)
    print("Vocabulary size:", len(vectorizer.vocabulary_))
    print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
    print("Processing time:", round(elapsed, 3), "seconds")

print("\nExample stemming cases:")
for word in ["organization", "organizer", "organizing",
             "organized", "organization's"]:
    print(word, "->", stemmer.stem(word))

README.md:   0%|          | 0.00/880 [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/2.87M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1225 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]


Method: Original
Vocabulary size: 20880
Accuracy: 0.9714
Processing time: 1.932 seconds

Method: Stemmed
Vocabulary size: 14615
Accuracy: 0.9592
Processing time: 1.16 seconds

Method: Lemmatized
Vocabulary size: 18725
Accuracy: 0.9714
Processing time: 1.431 seconds

Example stemming cases:
organization -> organ
organizer -> organ
organizing -> organ
organized -> organ
organization's -> organization'


In [10]:
import pandas as pd
from nltk.stem import PorterStemmer
from nltk.tokenize import RegexpTokenizer
from datasets import load_dataset

ps = PorterStemmer()
tokenizer = RegexpTokenizer(r"\w+")

# Fix: Load dataset from HF instead of expecting a local BBCNews.csv file
dataset = load_dataset("SetFit/bbc-news", split="train")
data = pd.DataFrame(dataset)
data = data.rename(columns={"text": "Text"})

data["Text"] = data["Text"].fillna("").astype(str)

def stem_text(text):
    words = tokenizer.tokenize(text.lower())
    return " ".join(ps.stem(word) for word in words)

data["Processed"] = data["Text"].apply(stem_text)

print("Original text:")
print(data["Text"].head())

print("\nStemmed text:")
print(data["Processed"].head())

# Compare at least 20 words
words = [
    "organization", "organizer", "organizing", "organized",
    "organization's", "running", "runner", "runs", "studies",
    "studied", "studying", "better", "businesses", "computers",
    "relational", "generously", "happiness", "connection",
    "connected", "connectivity"
]

print("\nOriginal Word -> Porter Stem")
for word in words:
    print(word, "->", ps.stem(word))

Original text:
0    wales want rugby league training wales could f...
1    china aviation seeks rescue deal scandal-hit j...
2    rock band u2 break ticket record u2 have smash...
3    markets signal brazilian recovery the brazilia...
4    tough rules for ringtone sellers firms that fl...
Name: Text, dtype: object

Stemmed text:
0    wale want rugbi leagu train wale could follow ...
1    china aviat seek rescu deal scandal hit jet fu...
2    rock band u2 break ticket record u2 have smash...
3    market signal brazilian recoveri the brazilian...
4    tough rule for rington seller firm that flout ...
Name: Processed, dtype: object

Original Word -> Porter Stem
organization -> organ
organizer -> organ
organizing -> organ
organized -> organ
organization's -> organization'
running -> run
runner -> runner
runs -> run
studies -> studi
studied -> studi
studying -> studi
better -> better
businesses -> busi
computers -> comput
relational -> relat
generously -> gener
happiness -> happi
connection

In [11]:

import nltk
from nltk.corpus import wordnet as wn

nltk.download("wordnet")
nltk.download("omw-1.4")

irregular = {
    "children": "child",
    "men": "man",
    "women": "woman",
    "mice": "mouse",
    "teeth": "tooth",
    "feet": "foot",
    "people": "person",
    "geese": "goose"
}

def parser(word):
    word = word.lower()

    if word in irregular:
        return irregular[word], "Irregular plural"

    # Use WordNet to check whether a candidate is a noun lemma
    candidate = wn.morphy(word, wn.NOUN)

    if candidate and candidate != word:
        return candidate, "Plural noun"

    if word.endswith("ies") and len(word) > 3:
        return word[:-3] + "y", "Plural noun (-ies)"

    if word.endswith(("ches", "shes", "xes", "zes", "sses")):
        return word[:-2], "Plural noun (-es)"

    if word.endswith("s") and not word.endswith(("ss", "us", "is")):
        return word[:-1], "Regular plural (-s)"

    return word, "Singular or unrecognized"

words = ["cars", "boxes", "cities", "children", "dogs",
         "mice", "books", "watches", "car"]

for word in words:
    print(word, "->", parser(word))


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


cars -> ('car', 'Plural noun')
boxes -> ('box', 'Plural noun')
cities -> ('city', 'Plural noun')
children -> ('child', 'Irregular plural')
dogs -> ('dog', 'Plural noun')
mice -> ('mouse', 'Irregular plural')
books -> ('book', 'Plural noun')
watches -> ('watch', 'Plural noun')
car -> ('car', 'Singular or unrecognized')


[nltk_data]   Package omw-1.4 is already up-to-date!
